# SmolVLM-256M: Full Benchmark

This notebook runs the full benchmark on one image. It compares the bare model, eager decoding and CUDA-graph decoding.

The benchmark also times the preprocessing steps and logs the RAM and VRAM. It uses a static cache of 1024 slots, because earlier tests showed that a small cache makes graph decoding faster.

The loop runs the benchmark once for each image processor option: `slow` (the original processor, on the CPU), `fast-cpu` (the fast processor, on the CPU) and `fast-gpu` (the fast processor, on the GPU). Each run loads the model again, so the cell takes some minutes.

Look at three things in the output: the time of the `make_inputs total` line (preprocessing), the prefill and decode times of the three modes, and the two answer checks.

In [18]:
REPO = "./edge-ai-smol-lvm-250-optimization"
MAX_LEN = 1024

In [19]:
import subprocess

for option in ["slow", "fast-cpu", "fast-gpu"]:
    r = subprocess.run(
        ["python", f"{REPO}/bench.py", "--image", f"{REPO}/frame.jpg",
         "--max-len", str(MAX_LEN), "--preprocess", option],
        capture_output=True, text=True,
    )
    print(f"=== preprocess: {option}")
    print(r.stdout if r.returncode == 0 else r.stderr.strip().splitlines()[-1])

=== preprocess: slow
preprocess: slow (Idefics3ImageProcessorPil, image on cpu)
memory after load: RAM peak 1822 MB | VRAM allocated 497 MB, peak 497 MB, reserved 520 MB
prompt tokens: 875 | max_len: 1024
preprocessing (JPEG decode and tokenize on CPU, tile + normalize on CPU):
  JPEG decode                                0.9 ms
  tile + normalize                         295.8 ms
  tokenize                                   0.1 ms
  make_inputs total (incl. copy to GPU)    314.3 ms
 bare: prefill  280.6 ms | decode 39.95 ms/token |   25.0 tok/s
eager: prefill  272.0 ms | decode 32.16 ms/token |   31.1 tok/s
graph: prefill  272.0 ms | decode  8.14 ms/token |  122.8 tok/s
identical outputs (eager vs graph): True
bare matches graph up to EOS: True
memory after benchmark: RAM peak 2228 MB | VRAM allocated 591 MB, peak 873 MB, reserved 1110 MB

=== preprocess: fast-cpu
preprocess: fast-cpu (Idefics3ImageProcessor, image on cpu)
memory after load: RAM peak 1823 MB | VRAM allocated 497 MB, pe

## Result

The test ran on a Colab GPU with one image. The prompt has 875 tokens, max_len is 1024, and the request makes 64 new tokens. All answer checks are True.

**Graph decoding, for each image processor option**

| Option | Image processor | Tile + normalize (ms) | Input total (ms) | Prefill (ms) | Decode (ms/token) | Request total (ms) | Preprocessing share |
|---|---|---|---|---|---|---|---|
| slow | Idefics3ImageProcessorPil | 295.8 | 314.3 | 272.0 | 8.14 | 1107 | 28% |
| fast-cpu | Idefics3ImageProcessor | 96.0 | 116.8 | 283.0 | 8.16 | 922 | 13% |
| fast-gpu | Idefics3ImageProcessor | 12.8 | 14.1 | 284.4 | 8.19 | 823 | 2% |

**Request total (ms) for each decoding mode**

| Mode | slow | fast-cpu | fast-gpu |
|---|---|---|---|
| bare | 3152 | 2877 | 2810 |
| eager | 2645 | 2405 | 2311 |
| graph | 1107 | 922 | 823 |

The request total is the input total, the prefill, and 64 decode steps.

- `fast-gpu` cuts the input preparation from 314 ms to 14 ms. With graph decoding, the request falls from 1107 ms to 823 ms (26% less).
- Graph decoding and `fast-gpu` together take the request from 3152 ms (bare model, `slow`) to 823 ms. This is 3.8 times faster.
- The fast processor on the CPU gives about half of the gain (1107 ms to 922 ms). The GPU gives the rest.
- Prefill is about 11 ms slower with both fast options (283 ms and 284 ms, against 272 ms). We do not know the cause. This is small against the 300 ms gain.
- Decode times do not depend on the image processor.
- Peak VRAM is 873 MB in all three runs. The prefill sets the peak, so GPU preprocessing does not raise it. The reserved memory is 22 MB higher with `fast-gpu` (1132 MB and 1110 MB). The earlier +146 MB result read the peak before the prefill ran.
- With `fast-gpu`, the request is now 64% decode (524 ms), 35% prefill (284 ms) and 2% preprocessing (14 ms). Decode is the next target.

In [ ]:
# best setup: fast-gpu preprocessing, graph decoding, with and without stop at EOS
import subprocess

r = subprocess.run(
    ["python", f"{REPO}/bench.py", "--image", f"{REPO}/frame.jpg",
     "--max-len", str(MAX_LEN), "--preprocess", "fast-gpu"],
    capture_output=True, text=True,
)
if r.returncode:
    print(r.stderr.strip().splitlines()[-1])
else:
    keys = ("make_inputs total", "graph:", "stop at EOS", "request")
    print("\n".join(l for l in r.stdout.splitlines() if any(k in l for k in keys)))

## Result: stop at EOS

The test ran on a Colab GPU with one image, `fast-gpu` preprocessing, graph decoding and max_len 1024. The answer has 9 tokens.

| Setup | Input (ms) | Prefill (ms) | Decode (ms) | Request total (ms) |
|---|---|---|---|---|
| 64 steps | 16.1 | 269.3 | 520 (64 x 8.13) | 806 |
| Stop at EOS (9 steps) | 16.1 | 271.8 | 80.9 | 369 |

- Stop at EOS cuts the request from 806 ms to 369 ms. This is 54% less, or 2.2 times faster.
- The result matches the EOS experiment: 80.9 ms for 9 steps against 79.2 ms.
- Decode takes about 9 ms for each step with only 9 steps. A fixed time of about 7 ms for each decode call is spread over few steps. A longer answer costs about 8.1 ms for each extra token.
- Prefill is now 74% of the request. Decode is 22% and the input is 4%.
- The first full request took 1107 ms (slow processor, 64 steps). It now takes 369 ms, which is 3.0 times faster.
- Prefill is the next target: fewer tiles, cached vision embeddings, and a TensorRT engine for SigLIP. Frame pipelining would hide only 16 ms of CPU work, so it now gains little.